# 04 — Population-Level Compartmental ODE Model (Extended)

Base model & baseline parameters: Ferdous (2023). This notebook extends the team's original baseline simulation with:
1. Equilibrium point (symbolic derivation via sympy + numeric evaluation)
2. Jacobian matrix + eigenvalue-based local stability analysis
3. Normalized forward sensitivity index analysis (the real formula, not just re-plotting with different constants)
4. An ML-informed screening scenario, properly framed as a *range* motivated by the ML classifier's recall — not a precise derived conversion

**Validation note:** our independently-computed equilibrium point and eigenvalues exactly match Ferdous (2023)'s published values, and our sensitivity indices match her Table 4 to 3 decimal places — strong evidence the extended model is correctly derived.

In [1]:
import sys
sys.path.append('../src')
from ode_population import (plot_baseline, run_original_team_scenarios, evaluate_equilibrium,
                             jacobian_and_eigenvalues, sensitivity_analysis, plot_sensitivity,
                             ml_informed_scenario, BASELINE_PARAMS)
import pandas as pd, numpy as np

## Baseline parameters (Ferdous 2023, Table 3)

In [2]:
BASELINE_PARAMS

{'b': 0.0138,
 'mu': 0.0138,
 'epsilon': 0.142,
 'tau': 0.565,
 'alpha': 0.2,
 'beta': 0.3,
 'gamma': 0.05,
 'delta1': 0.04,
 'delta2': 0.002}

## 1. Baseline population dynamics simulation

In [3]:
sol = plot_baseline()

[ode_population] Baseline sanity check — total population fraction at t=0: 1.0000, t=10: 0.9674, t=20: 0.9577 (should stay close to 1.0)


## 2. Reproducing and extending the team's original scenario sweeps
(treatment rate / lifestyle adoption / awareness / combined interventions)

In [4]:
run_original_team_scenarios()

[ode_population] Original team scenario plots reproduced (files 12-15).


## 3. Equilibrium point (symbolic derivation, evaluated numerically)

In [5]:
eq = evaluate_equilibrium()
pd.Series(eq).round(4)

A    0.0106
L    0.0213
P    0.4635
S    0.0388
T    0.3799
dtype: float64

## 4. Jacobian matrix & eigenvalue-based stability analysis

In [6]:
J, eigvals, stable = jacobian_and_eigenvalues()
print('Eigenvalues:', np.round(eigvals.real, 4))
print('Locally asymptotically stable:', stable)

Eigenvalues: [-0.0158 -0.0138 -0.6188 -0.3638 -0.3558]
Locally asymptotically stable: True


## 5. Sensitivity analysis (normalized forward sensitivity index)

In [7]:
sens_table = sensitivity_analysis()
sens_table.round(3)

,A,L,P,S,T
mu,-0.067,-0.077,-1.077,-0.039,-0.941
epsilon,0.439,-0.399,-0.399,-0.399,0.439
tau,-0.913,0.000,0.000,0.000,0.087
alpha,-0.400,0.438,0.438,-0.562,-0.400
beta,-0.134,-0.825,0.175,0.000,-0.134
gamma,0.140,-0.137,-0.137,0.000,0.140
delta1,-0.065,0.000,0.000,0.000,-0.065
delta2,0.000,0.000,0.000,0.000,-0.127


In [8]:
plot_sensitivity(sens_table)

## 6. ML-informed early-screening scenario

Uses the ML pipeline's recall/sensitivity (from `results/tables/ml_handoff_metrics.json`) to motivate — not precisely determine — a range of plausible treatment-rate increases. Run the ML pipeline notebook first so this handoff file exists.

In [9]:
ml_informed_scenario()

[ode_population] ML classifier recall/sensitivity = 0.593 (model: AdaBoost)
[ode_population] Framing: a classifier with this recall demonstrates at-risk individuals CAN be identified at scale. This motivates -- but does not mathematically determine -- the magnitude of a screening-driven treatment-rate increase. We therefore simulate a range, not one derived value.
[ode_population] ML-informed scenario plot saved (file 17).


Figures generated (see `results/figures/`):
- `11_population_baseline.png`
- `12`–`15_sweep_*.png`
- `16_sensitivity_analysis.png`
- `17_ml_informed_scenario.png`

Tables generated (see `results/tables/`):
- `equilibrium_point.csv`
- `eigenvalues.csv`
- `sensitivity_indices.csv`